# 03 · Evaluate the classifier

Test-set accuracy, precision/recall/F1, ROC-AUC, per-manipulation AUC, video-level scores, and the model's most confident mistakes.

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 x2 (not P100)
- **Internet:** On
- **Inputs:** output of **01_extract_faces**; output of **02_train**

**Produces:** `/kaggle/working/outputs/metrics/` (JSON, plots, per-image predictions). When it finishes, use **Save Version → Save & Run All**, then add this notebook's output as an input to the next notebook.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Uses the repo if it is attached as an input (zip uploaded as a Kaggle dataset, or an earlier
# notebook's output); otherwise clones it. For a private repo add a GITHUB_TOKEN secret.
import os, sys, glob, shutil, subprocess

PROJECT = "/kaggle/working/project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
    if found:
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
        if r.returncode:
            raise RuntimeError("git clone failed: attach the project as a dataset or add a GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
print("project ready:", sorted(os.listdir(PROJECT)))

In [ ]:
from src.kaggle_utils import setup
setup(data=True, checkpoint=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

In [ ]:
!python -m src.classifier.evaluate --workers 4
metrics = json.load(open("/kaggle/working/outputs/metrics/test_metrics.json"))
display(pd.Series({k: v for k, v in metrics.items() if not isinstance(v, dict)}).round(4).to_frame("test"))
display(pd.Series(metrics.get("video_level", {})).round(4).to_frame("video level"))

In [ ]:
from IPython.display import Image as Show
display(Show("/kaggle/working/outputs/metrics/test_roc.png", width=420),
        Show("/kaggle/working/outputs/metrics/test_confusion.png", width=420))

## Per-manipulation performance
Each method's fakes vs. all real test frames.

In [ ]:
pm = pd.DataFrame(metrics.get("per_method", {})).T
display(pm.round(4))
pm.plot.bar(figsize=(8, 3.5), ylim=(0, 1), rot=0, title="Per-method test performance"); plt.show()

## Most confident mistakes

In [ ]:
import torch
from pathlib import Path
from src.classifier.dataset import make_loader
from src.classifier.model import load_model
from src.classifier.train import predict
from src.config import CHECKPOINT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = make_loader("test", 64, num_workers=4)
probs, labels = predict(load_model(CHECKPOINT, device), loader, device)
preds = pd.DataFrame({"path": [p for p, _ in loader.dataset.samples], "label": labels, "prob_fake": probs})
preds["method"] = preds.path.map(lambda p: Path(p).name.split("__")[0])
preds.to_csv("/kaggle/working/outputs/metrics/test_predictions.csv", index=False)

thr = metrics["threshold"]  # tuned on validation
wrong = preds[(preds.prob_fake > thr) != (preds.label == 1)]
worst = pd.concat([wrong[wrong.label == 0].nlargest(6, "prob_fake"), wrong[wrong.label == 1].nsmallest(6, "prob_fake")])
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, r in zip(axes.flat, worst.itertuples()):
    ax.imshow(Image.open(r.path)); ax.axis("off")
    ax.set_title(f"{'real→FAKE' if r.label == 0 else r.method + '→REAL'}\nP(fake)={r.prob_fake:.2f}", fontsize=8)
plt.suptitle("Top: real frames called fake · Bottom: fakes called real"); plt.tight_layout(); plt.show()
print(f"{len(wrong)} / {len(preds)} test frames misclassified")